# Phần 1: Giải 10 Câu Trắc Nghiệm

---

## Mục tiêu

Phần này giải quyết 10 câu hỏi trắc nghiệm bằng cách truy vấn trực tiếp trên bộ dữ liệu được cung cấp. Mỗi câu được trình bày theo cấu trúc:

1. **Đề bài** — phát biểu lại câu hỏi và các lựa chọn A/B/C/D
2. **Phương pháp** — giải thích logic và các bước tính toán
3. **Code** — mã Python chạy trên dữ liệu
4. **Kết luận** — đáp án chính xác được xác nhận từ output

**Lưu ý về dữ liệu:** Toàn bộ 10 câu đều được tính toán trực tiếp từ các file CSV gốc (raw data) mà không qua bước EDA hay làm sạch trước. Lý do là các câu hỏi chỉ yêu cầu tổng hợp, đếm, tính trung bình — các phép tính này không bị ảnh hưởng bởi missing values hay outlier ở mức cần xử lý trước. Bất kỳ bước làm sạch nào cần thiết (ép kiểu, fillna, filter) đều được thực hiện bên trong chính cell của câu đó.

---
## Setup: Import thư viện và đọc dữ liệu

In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
customers = pd.read_csv(os.path.join(path, "customers.csv"))
geography = pd.read_csv(os.path.join(path, "geography.csv"))
inventory = pd.read_csv(os.path.join(path, "inventory.csv"))
order_items = pd.read_csv(os.path.join(path, "order_items.csv"))
orders = pd.read_csv(os.path.join(path, "orders.csv"))
payments = pd.read_csv(os.path.join(path, "payments.csv"))
products = pd.read_csv(os.path.join(path, "products.csv"))
promotions = pd.read_csv(os.path.join(path, "promotions.csv"))
returns = pd.read_csv(os.path.join(path, "returns.csv"))
reviews = pd.read_csv(os.path.join(path, "reviews.csv"))
sales = pd.read_csv(os.path.join(path, "sales.csv"))
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))
shipments = pd.read_csv(os.path.join(path, "shipments.csv"))
web_traffic = pd.read_csv(os.path.join(path, "web_traffic.csv"))

print("Load xong tất cả DataFrame")

Load xong tất cả DataFrame


In [ ]:
# Kiểm tra kích thước các DataFrame
for name, df in {
    "customers": customers,
    "geography": geography,
    "inventory": inventory,
    "order_items": order_items,
    "orders": orders,
    "payments": payments,
    "products": products,
    "promotions": promotions,
    "returns": returns,
    "reviews": reviews,
    "sales": sales,
    "sample_submission": sample_submission,
    "shipments": shipments,
    "web_traffic": web_traffic
}.items():
    print(f"{name:<20} {df.shape}")

customers            (121930, 7)
geography            (39948, 4)
inventory            (60247, 17)
order_items          (714669, 7)
orders               (646945, 8)
payments             (646945, 4)
products             (2412, 8)
promotions           (50, 10)
returns              (39939, 7)
reviews              (113551, 7)
sales                (3833, 3)
sample_submission    (548, 3)
shipments            (566067, 4)
web_traffic          (3652, 7)


---
## Q1 — Trung vị khoảng cách giữa các lần mua

**Đề bài:** Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ `orders.csv`)

| Lựa chọn | Giá trị |
|----------|---------|
| A | 30 ngày |
| B | 90 ngày |
| **C** | **144 ngày** |
| D | 365 ngày |

**Phương pháp:**

1. Sắp xếp `orders` theo `customer_id`, `order_date`, `order_id` để đảm bảo thứ tự thời gian chính xác khi có nhiều đơn trong cùng một ngày.
2. Lọc chỉ giữ lại các khách hàng có nhiều hơn một đơn hàng bằng `.filter(lambda x: len(x) > 1)` — đúng với yêu cầu đề bài.
3. Dùng `groupby("customer_id") + shift()` để lấy `order_date` của đơn liền kề trước đó của cùng khách hàng. Dòng đầu tiên của mỗi khách sẽ là NaN.
4. Tính chênh lệch ngày giữa đơn hiện tại và đơn trước (`inter_order_gap`).
5. Bỏ NaN bằng `.dropna()` rồi lấy `median()`.

In [ ]:
# Q1
q1_df = orders.sort_values(["customer_id", "order_date", "order_id"]).copy()
q1_df["order_date"] = pd.to_datetime(q1_df["order_date"])

# Chỉ lấy khách hàng có nhiều hơn 1 đơn hàng
khach_nhieu_don = (
    q1_df.groupby("customer_id")
    .filter(lambda x: len(x) > 1)
    .copy()
)

# Tính khoảng cách giữa hai lần mua liên tiếp bằng shift()
khach_nhieu_don["ngay_don_truoc"] = (
    khach_nhieu_don.groupby("customer_id")["order_date"].shift()
)
khach_nhieu_don["inter_order_gap"] = (
    khach_nhieu_don["order_date"] - khach_nhieu_don["ngay_don_truoc"]
).dt.days

q1_result = khach_nhieu_don["inter_order_gap"].dropna().median()
print(f"Trung vị khoảng cách giữa hai lần mua: {q1_result} ngày")

Trung vị khoảng cách giữa hai lần mua: 144.0 ngày


**Kết luận Q1: Đáp án C — 144 ngày**

Trung vị khoảng cách giữa hai lần mua liên tiếp của cùng một khách hàng là 144 ngày (khoảng 4.8 tháng). Con số này hợp lý với một doanh nghiệp thời trang thương mại điện tử: khách hàng mua khá đều đặn nhưng không quá thường xuyên, phù hợp với chu kỳ thay đổi trang phục theo mùa.

---
## Q2 — Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất

**Đề bài:** Phân khúc sản phẩm (segment) nào trong `products.csv` có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức `(price - cogs) / price`?

| Lựa chọn | Giá trị |
|----------|---------|
| A | Premium |
| B | Performance |
| C | Activewear |
| **D** | **Standard** |

**Phương pháp:**

1. Tính `gross_margin` cho từng sản phẩm theo công thức của đề bài: `(price - cogs) / price`.
2. Nhóm theo `segment` và tính `mean()` của `gross_margin` trong từng nhóm.
3. Sắp xếp giảm dần và lấy `idxmax()` để xác định segment có margin cao nhất.

Lưu ý: Đề bài hỏi **gross margin trung bình**, tức là trung bình của hệ số margin từng sản phẩm — không phải tính tổng revenue rồi chia, do đó mỗi sản phẩm trong cùng một segment có trọng số ngang nhau bất kể doanh số.

In [ ]:
# Q2
q2_df = products.copy()
q2_df["gross_margin"] = (q2_df["price"] - q2_df["cogs"]) / q2_df["price"]

q2_result = q2_df.groupby("segment")["gross_margin"].mean().sort_values(ascending=False)
print(q2_result)
print(f"\nPhân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất: {q2_result.idxmax()}")

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64

Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất: Standard


**Kết luận Q2: Đáp án D — Standard**

Phân khúc Standard có gross margin trung bình là 31.3%, cao nhất trong tất cả 8 phân khúc. Đây cũng là insight kinh doanh đáng chú ý: sản phẩm Standard (giá phải chăng) lại có biên lợi nhuận cao hơn cả Premium (28.5%), cho thấy chi phí sản xuất phân khúc này được kiểm soát hiệu quả hơn.

---
## Q3 — Lý do trả hàng phổ biến nhất trong Streetwear

**Đề bài:** Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join `returns` với `products` theo `product_id`), lý do trả hàng nào xuất hiện nhiều nhất?

| Lựa chọn | Giá trị |
|----------|---------|
| A | defective |
| **B** | **wrong_size** |
| C | changed_mind |
| D | not_as_described |

**Phương pháp:**

1. Join bảng `returns` với bảng `products` theo `product_id`, lấy thêm cột `category`.
2. Lọc chỉ giữ các bản ghi có `category == "Streetwear"`.
3. Đếm tần suất xuất hiện của từng giá trị trong cột `return_reason` bằng `value_counts()`.
4. Lấy `idxmax()` để xác định lý do có count cao nhất.

In [ ]:
# Q3
q3_df = returns.merge(products[["product_id", "category"]], on="product_id")
q3_df = q3_df[q3_df["category"] == "Streetwear"]

q3_result = q3_df["return_reason"].value_counts()
print(q3_result)
print(f"\nLý do trả hàng xuất hiện nhiều nhất: {q3_result.idxmax()}")

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

Lý do trả hàng xuất hiện nhiều nhất: wrong_size


**Kết luận Q3: Đáp án B — wrong_size**

wrong_size chiếm 7,626 bản ghi — gấp đôi so với lý do đứng thứ hai (defective: 4,330). Đối với thời trang Streetwear, việc khách hàng mua online mà không thử được size là điều khó tránh, đặc biệt khi các thương hiệu hay dùng sizing khác nhau. Đây là insight quan trọng cho việc tối ưu hóa size guide và chính sách đổi trả.

---
## Q4 — Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất

**Đề bài:** Trong `web_traffic.csv`, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

| Lựa chọn | Giá trị |
|----------|---------|
| A | organic_search |
| B | paid_search |
| **C** | **email_campaign** |
| D | social_media |

**Phương pháp:**

1. Nhóm `web_traffic` theo `traffic_source`.
2. Tính `mean()` của `bounce_rate` trong từng nhóm — đây là trung bình bounce rate qua tất cả các ngày có nguồn đó xuất hiện.
3. Sắp xếp tăng dần và lấy `idxmin()` để tìm nguồn có bounce rate trung bình thấp nhất.

In [ ]:
# Q4
q4_df = web_traffic.copy()
q4_result = q4_df.groupby("traffic_source")["bounce_rate"].mean().sort_values()
print(q4_result)
print(f"\nNguồn truy cập có tỷ lệ thoát trung bình thấp nhất: {q4_result.idxmin()}")

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: email_campaign


**Kết luận Q4: Đáp án C — email_campaign**

email_campaign có bounce rate trung bình thấp nhất (0.00446). Điều này hợp lý: người dùng đến từ email campaign đã biết mình muốn xem gì (họ chủ động mở email và click), nên ít có khả năng thoát ngay sau khi vào trang. Ngược lại, organic search hay direct có thể đem lại nhiều lượng truy cập ngẫu nhiên hơn.

---
## Q5 — Tỷ lệ dòng order_items có áp dụng khuyến mãi

**Đề bài:** Tỷ lệ phần trăm các dòng trong `order_items.csv` có áp dụng khuyến mãi (tức là `promo_id` không null) xấp xỉ là bao nhiêu?

| Lựa chọn | Giá trị |
|----------|---------|
| A | 12% |
| B | 25% |
| **C** | **39%** |
| D | 54% |

**Phương pháp:**

1. Kiểm tra cột `promo_id` trong `order_items` xem có null hay không bằng `.notna()` — trả về True/False cho từng dòng.
2. `.mean()` trên series Boolean sẽ trả về tỷ lệ dòng có giá trị True (tức là promo_id không null).
3. Nhân 100 để đổi sang phần trăm.

Lưu ý: Đề bài chỉ hỏi `promo_id`, không hỏi `promo_id_2`. Việc có `promo_id` là null hay không là điều kiện đủ để xác định dòng đó có áp dụng khuyến mãi hay không.

In [ ]:
# Q5
q5_result = order_items["promo_id"].notna().mean() * 100
print(f"Tỷ lệ dòng có áp dụng khuyến mãi: {q5_result:.2f}%")

Tỷ lệ dòng có áp dụng khuyến mãi: 38.66%


**Kết luận Q5: Đáp án C — 39%**

38.66% các dòng order_items có áp dụng khuyến mãi, xấp xỉ 39%. Con số này cho thấy khoảng 2/5 giao dịch sản phẩm được thực hiện kèm khuyến mãi — mức độ khá cao, phản ánh chiến lược khuyến mãi tích cực của doanh nghiệp.

---
## Q6 — Nhóm tuổi có số đơn hàng trung bình cao nhất

**Đề bài:** Trong `customers.csv`, xét các khách hàng có `age_group` khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

| Lựa chọn | Giá trị |
|----------|---------|
| **A** | **55+** |
| B | 25-34 |
| C | 35-44 |
| D | 45-54 |

**Phương pháp:**

1. Từ `orders`, đếm số đơn hàng của từng khách hàng bằng `groupby("customer_id").count()`.
2. Left-join vào `customers` (đã lọc bỏ null age_group) để gán số đơn cho từng khách hàng. Dùng `how="left"` để giữ lại cả những khách hàng chưa đặt đơn nào (họ sẽ được điền 0 vào số đơn hàng).
3. Nhóm theo `age_group`, tính `mean()` của số đơn hàng. Đây chính là số đơn hàng trung bình trên mỗi khách hàng trong nhóm đó.
4. Lấy nhóm có giá trị cao nhất bằng `idxmax()`.

In [ ]:
# Q6
q6_so_don = orders.groupby("customer_id")["order_id"].count().reset_index()
q6_so_don.columns = ["customer_id", "so_don_hang"]

q6_df = customers[customers["age_group"].notna()].merge(q6_so_don, on="customer_id", how="left")
q6_df["so_don_hang"] = q6_df["so_don_hang"].fillna(0)

q6_result = q6_df.groupby("age_group")["so_don_hang"].mean().sort_values(ascending=False)
print(q6_result)
print(f"\nNhóm tuổi có số đơn hàng trung bình cao nhất: {q6_result.idxmax()}")

age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: so_don_hang, dtype: float64

Nhóm tuổi có số đơn hàng trung bình cao nhất: 55+


**Kết luận Q6: Đáp án A — 55+**

Nhóm 55+ có trung bình 5.41 đơn hàng mỗi khách — cao nhất trong tất cả các nhóm tuổi. Đáng chú ý là sự chênh lệch giữa các nhóm không quá lớn (từ 5.22 đến 5.41), cho thấy hành vi mua hàng khá đồng đều. Tuy nhiên nhóm cao tuổi hơn lại có xu hướng trung thành và mua nhiều hơn.

---
## Q7 — Vùng tạo ra tổng doanh thu cao nhất

**Đề bài:** Vùng (region) nào trong `geography.csv` tạo ra tổng doanh thu cao nhất trong `sales_train.csv`?

| Lựa chọn | Giá trị |
|----------|---------|
| A | West |
| B | Central |
| **C** | **East** |
| D | Cả ba vùng có doanh thu xấp xỉ bằng nhau |

**Phương pháp:**

`sales.csv` chỉ có cột Date/Revenue/COGS mà không có thông tin địa lý. Để tính doanh thu theo vùng, cần truy vết ngược qua các bảng:

1. Tính `doanh_thu` từng dòng trong `order_items`: `unit_price * quantity - discount_amount`.
2. Join `order_items` với `orders` theo `order_id` để lấy `zip` của địa chỉ giao hàng.
3. Join tiếp với `geography` theo `zip` để lấy `region`.
4. Nhóm theo `region`, tính `sum()` của `doanh_thu`, sắp xếp giảm dần.

Cách tính này dùng `zip` từ `orders` (địa chỉ giao hàng) làm đại diện cho vùng tiêu thụ — phù hợp nhất với ý nghĩa kinh doanh.

In [ ]:
# Q7 — tính doanh thu theo vùng vì sales.csv không có cột region
q7_df = order_items.copy()
q7_df["doanh_thu"] = q7_df["unit_price"] * q7_df["quantity"] - q7_df["discount_amount"]
q7_df = q7_df.merge(orders[["order_id", "zip"]], on="order_id")
q7_df = q7_df.merge(geography[["zip", "region"]], on="zip")

q7_result = q7_df.groupby("region")["doanh_thu"].sum().sort_values(ascending=False)
print(q7_result)
print(f"\nVùng tạo ra tổng doanh thu cao nhất: {q7_result.idxmax()}")

region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: doanh_thu, dtype: float64

Vùng tạo ra tổng doanh thu cao nhất: East


**Kết luận Q7: Đáp án C — East**

Vùng East dẫn đầu với ~7.29 tỷ đồng doanh thu, gấp 1.5 lần vùng Central (~4.72 tỷ) và gần gấp đôi vùng West (~3.67 tỷ). Vùng East ở Việt Nam bao gồm Đông Nam Bộ (TP.HCM, Bình Dương, Đồng Nai) — trung tâm thương mại lớn nhất cả nước, nên việc dẫn đầu về doanh thu là hoàn toàn hợp lý.

---
## Q8 — Phương thức thanh toán nhiều nhất trong đơn bị hủy

**Đề bài:** Trong các đơn hàng có `order_status = 'cancelled'` trong `orders.csv`, phương thức thanh toán nào được sử dụng nhiều nhất?

| Lựa chọn | Giá trị |
|----------|---------|
| **A** | **credit_card** |
| B | cod |
| C | paypal |
| D | bank_transfer |

**Phương pháp:**

1. Lọc `orders` chỉ lấy các dòng có `order_status == "cancelled"`.
2. Đếm tần suất của từng giá trị trong cột `payment_method` bằng `value_counts()`.
3. Lấy phương thức có count cao nhất bằng `idxmax()`.

Lưu ý: `payment_method` ở đây lấy từ bảng `orders`, không phải từ `payments`. Theo schema, hai bảng này có quan hệ 1:1 nên kết quả tương đương — nhưng thông tin trong `orders` là đủ để trả lời câu này.

In [ ]:
# Q8
q8_df = orders[orders["order_status"] == "cancelled"]
q8_result = q8_df["payment_method"].value_counts().sort_values(ascending=False)
print(q8_result)
print(f"\nPhương thức thanh toán được sử dụng nhiều nhất trong đơn hủy: {q8_result.idxmax()}")

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

Phương thức thanh toán được sử dụng nhiều nhất trong đơn hủy: credit_card


**Kết luận Q8: Đáp án A — credit_card**

credit_card chiếm 28,452 đơn bị hủy — nhiều nhất, nhưng điều này chủ yếu phản ánh credit_card cũng là phương thức thanh toán phổ biến nhất tổng thể (tỷ trọng lớn nhất). Để có insight sâu hơn cần tính tỷ lệ hủy theo từng phương thức thanh toán, nhưng với câu hỏi này đề bài chỉ hỏi về số lượng tuyệt đối.

---
## Q9 — Kích thước có tỷ lệ trả hàng cao nhất

**Đề bài:** Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong `returns` chia cho số dòng trong `order_items` (join với `products` theo `product_id`)?

| Lựa chọn | Giá trị |
|----------|---------|
| **A** | **S** |
| B | M |
| C | L |
| D | XL |

**Phương pháp:**

1. Join `order_items` với `products` theo `product_id` để lấy cột `size` — dùng để đếm tổng số dòng bán của từng size.
2. Tương tự, join `returns` với `products` để lấy `size` của từng bản ghi trả hàng.
3. Đếm số dòng trong mỗi bảng theo `size`.
4. Tính tỷ lệ: `số trả / số bán` theo từng size, chỉ xét 4 size S/M/L/XL theo đề bài.
5. Lấy size có tỷ lệ cao nhất.

In [ ]:
# Q9
q9_items = order_items.merge(products[["product_id", "size"]], on="product_id")
q9_returns = returns.merge(products[["product_id", "size"]], on="product_id")

# Đếm số dòng bán và số dòng trả theo kích thước
q9_so_ban = q9_items.groupby("size")["order_id"].count()
q9_so_tra = q9_returns.groupby("size")["return_id"].count()

# Chỉ giữ 4 kích thước chính theo đề bài
bon_kich_thuoc = ["S", "M", "L", "XL"]
q9_result = (
    (q9_so_tra / q9_so_ban)
    .loc[bon_kich_thuoc]
    .sort_values(ascending=False)
)
print(q9_result)
print(f"\nKích thước có tỷ lệ trả hàng cao nhất: {q9_result.idxmax()}")

size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64

Kích thước có tỷ lệ trả hàng cao nhất: S


**Kết luận Q9: Đáp án A — S**

Size S có tỷ lệ trả hàng cao nhất (5.65%), tuy chênh lệch với các size khác rất nhỏ (5.52% - 5.65%). Điều này có thể do khách hàng nhỏ con (size S) khó ước tính size hơn khi mua online, hoặc sizing của size S không nhất quán giữa các sản phẩm trong danh mục.

---
## Q10 — Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất

**Đề bài:** Trong `payments.csv`, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

| Lựa chọn | Giá trị |
|----------|---------|
| A | 1 kỳ (trả một lần) |
| B | 3 kỳ |
| **C** | **6 kỳ** |
| D | 12 kỳ |

**Phương pháp:**

1. Nhóm `payments` theo `installments` (số kỳ trả góp).
2. Tính `mean()` của `payment_value` trong từng nhóm — đây là giá trị đơn hàng trung bình mà khách chọn số kỳ tương ứng.
3. Sắp xếp giảm dần và lấy số kỳ có trung bình cao nhất.

In [ ]:
# Q10
q10_result = payments.groupby("installments")["payment_value"].mean().sort_values(ascending=False)
print(q10_result)
print(f"\nKế hoạch trả góp có giá trị thanh toán trung bình cao nhất: {q10_result.idxmax()} kỳ")

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64

Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất: 6 kỳ


**Kết luận Q10: Đáp án C — 6 kỳ**

Kế hoạch 6 kỳ có payment_value trung bình cao nhất (~24,447 đồng), chỉ cao hơn chút ít so với 3 kỳ (~24,400) và 12 kỳ (~24,246). Đáng chú ý là kỳ hạn 2 kỳ có giá trị rất thấp (~708 đồng) — có thể đây là một tùy chọn ít được sử dụng hoặc chỉ áp dụng cho đơn hàng giá trị nhỏ. Sự chênh lệch nhỏ giữa 1/3/6/12 kỳ cho thấy chính sách trả góp khá linh hoạt, không phân biệt nhiều về mức đơn hàng.

---
## Tổng kết đáp án Phần 1

| Câu | Đáp án | Kết quả từ dữ liệu |
|-----|--------|-------------------|
| Q1 | C — 144 ngày | 144.0 ngày |
| Q2 | D — Standard | gross margin 0.3134 |
| Q3 | B — wrong_size | 7,626 bản ghi |
| Q4 | C — email_campaign | bounce rate 0.00446 |
| Q5 | C — 39% | 38.66% |
| Q6 | A — 55+ | 5.41 đơn/khách |
| Q7 | C — East | ~7.29 tỷ đồng |
| Q8 | A — credit_card | 28,452 đơn hủy |
| Q9 | A — S | tỷ lệ 5.65% |
| Q10 | C — 6 kỳ | ~24,447 đồng/đơn |